# 02 — Data-quality proofs (evidence, not claims)
Every number quoted in the report, reproduced here from `../data/`. Run from the `notebooks/` directory.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
tr = pd.read_csv('../data/train_test.csv', parse_dates=['date'])
va = pd.read_csv('../data/validation.csv', parse_dates=['date'])
print('train', tr.shape, tr['date'].min().date(), '->', tr['date'].max().date())
print('val  ', va.shape, va['date'].min().date(), '->', va['date'].max().date())

train (48000, 14) 2025-01-01 -> 2025-10-31
val   (12000, 13) 2025-11-01 -> 2025-12-31


## 1. Weight cap at 47,500 (pile-up proof)

In [2]:
w = tr['weight']
print('exactly +47500:', (w == 47500).sum(), '| exactly -47500:', (w == -47500).sum())
print('47000-47499:', ((w.abs() >= 47000) & (w.abs() < 47500)).sum())
print('max:', w.max(), '| min:', w.min())
print(w.value_counts().head(5).to_string())

exactly +47500: 1191 | exactly -47500: 13
47000-47499: 175
max: 47500.0 | min: -47500.0
weight
 47500.0    1191
 5000.0       31
-47500.0      13
 32436.0      10
 32153.0      10


## 2. Distance floor at 70.0

In [3]:
d = tr['distance']
print('exactly 70.0:', (d == 70.0).sum(), '| min:', d.min())
print('70<d<80:', ((d > 70.0) & (d < 80.0)).sum(), '| 80<=d<100:', ((d >= 80.0) & (d < 100.0)).sum())
print(d.value_counts().head(5).to_string())

exactly 70.0: 48 | min: 70.0
70<d<80: 74 | 80<=d<100: 150
distance
70.0     48
446.3    12
741.1    11
758.9    11
429.3    10


## 3. Negative weights, missing values, market drift

In [4]:
print('neg weight train/val:', (tr['weight'] < 0).sum(), (va['weight'] < 0).sum())
print('weight NaN train/val:', tr['weight'].isna().sum(), va['weight'].isna().sum())
print('market NaN train/val:', tr['market_index'].isna().sum(), va['market_index'].isna().sum())
print('market mean train/val: %.3f / %.3f' % (tr['market_index'].mean(), va['market_index'].mean()))
tr_c = set(tr['pickup']) | set(tr['delivery']); va_c = set(va['pickup']) | set(va['delivery'])
print('unseen val cities:', sorted(va_c - tr_c))

neg weight train/val: 292 145
weight NaN train/val: 300 165
market NaN train/val: 374 249
market mean train/val: 1.083 / 0.927
unseen val cities: ['Allentown', 'Charlotte', 'Chicago', 'Jackson', 'Knoxville', 'Laredo', 'Norfolk', 'San Diego']


## 4. quote_signal U-shape vs $/mile + market weekly cycle

In [5]:
tr['rpm'] = tr['posted_rate'] / tr['distance']
print(tr.groupby(pd.cut(tr['quote_signal'], 8), observed=True)['rpm'].mean().round(2).to_string())
tr['dow'] = tr['date'].dt.dayofweek
print('\nmarket by dow (Mon=0):')
print(tr.groupby('dow')['market_index'].mean().round(3).to_string())
print('\nrpm by dow:')
print(tr.groupby('dow')['rpm'].mean().round(3).to_string())
print('\ncorr distance-rate: %.3f' % tr['distance'].corr(tr['posted_rate']))

quote_signal
(0.689, 1.057]    3.20
(1.057, 1.422]    2.88
(1.422, 1.787]    2.34
(1.787, 2.151]    2.11
(2.151, 2.516]    2.19
(2.516, 2.881]    2.70
(2.881, 3.246]    3.01
(3.246, 3.61]     3.39

market by dow (Mon=0):
dow
0    1.000
1    1.075
2    1.153
3    1.179
4    1.134
5    1.047
6    0.989

rpm by dow:
dow
0    2.196
1    2.213
2    2.248
3    2.245
4    2.215
5    2.196
6    2.192

corr distance-rate: 0.909
